# Notebook 11 — Hourly agent heartbeat and crash supervision

Tagged definitions only. Inputs: immutable original12hour clock/config and validatedCLIauth. Outputs30secondheartbeat, hourly agent review logs, recoverynotebooks and exactcheckpointresume receipts. Never changes scientific endpoints or frozenexports; keeps one writer and the original deadline. Main Notebook11RunAll starts detachedsupervisor; importing does not start it. Stops whenstudycomplete/incomplete/deadline. Agent may safelyfix operationalbugs but cannot guarantee recovery from everypossiblefailure.

In [ ]:

"""Hourly agent reviews and bounded exact-checkpoint restart supervision."""
from pathlib import Path
from datetime import datetime,timezone
import hashlib,importlib.util,json,os,shlex,signal,subprocess,sys,time
import nbformat
from fmm_jepa.bootstrap import TaggedNotebookLoader
from fmm_jepa.training import atomic_json,digest_file,run_writer
CLI=Path('/home/benjamin/.vscode-server/extensions/openai.chatgpt-26.5930.61225-linux-x64/bin/linux-x86_64/codex')
TERMINAL={'COMPLETED','INCOMPLETE'}
def _read(p):
    p=Path(p);return json.loads(p.read_text()) if p.exists() else {}
def _exists(session):
    return subprocess.run(['tmux','has-session','-t','='+session],capture_output=True).returncode==0
def _workflow(root):
    key='fmm11_workflow'
    if key not in sys.modules:
        spec=importlib.util.spec_from_loader(key,TaggedNotebookLoader(Path(root)/'experiments/11_manifold_long/workflow.ipynb'))
        module=importlib.util.module_from_spec(spec);sys.modules[key]=module;spec.loader.exec_module(module)
    return sys.modules[key]
def snapshot(root,run_id):
    root=Path(root);artifact=root/'artifacts/notebook_11'/run_id
    clock=_read(artifact/'budget_clock.json');status=_read(artifact/'study_status.json')
    worker='fmm-manifold-long-'+run_id
    rows=[];activity=float(status.get('updated_unix',clock.get('started_unix',0)))
    for p in sorted((root/'checkpoints'/run_id/'stages/learning/arms').glob('*/manifest.json')):
        m=_read(p);rows.append({'arm':p.parent.name,'successful_updates':m.get('successful_updates',0)})
        try:activity=max(activity,p.stat().st_mtime)
        except FileNotFoundError:pass
    for p in (artifact/'stages/learning').rglob('*.json'):
        try:activity=max(activity,p.stat().st_mtime)
        except FileNotFoundError:pass
    for p in (root/'checkpoints'/run_id/'stages/learning/probes').rglob('probe_manifest.json'):
        try:activity=max(activity,p.stat().st_mtime)
        except FileNotFoundError:pass
    gpu=subprocess.run(['nvidia-smi','--query-gpu=index,utilization.gpu,memory.used,memory.total',
        '--format=csv,noheader,nounits'],capture_output=True,text=True,timeout=15)
    return {'observed_unix':time.time(),'status':status,'worker_session':worker,'worker_alive':_exists(worker),
        'clock':clock,'remaining_seconds':max(0,clock.get('deadline_unix',time.time())-time.time()),
        'last_activity_unix':activity,'activity_age_seconds':max(0,time.time()-activity),
        'training_progress':rows,'evaluation_progress':_read(artifact/'stages/learning/evaluation_status.json'),
        'gpu_csv':gpu.stdout.strip(),'gpu_query_returncode':gpu.returncode}
def agent_review(root,run_id,event='hourly'):
    root=Path(root);artifact=root/'artifacts/notebook_11'/run_id
    snap=snapshot(root,run_id);remaining=snap['remaining_seconds']
    if remaining<30:return {'status':'SKIPPED_NEAR_DEADLINE','event':event}
    folder=artifact/'heartbeat'/'agent_reviews'/str(int(time.time()));folder.mkdir(parents=True,exist_ok=True)
    schema={'type':'object','properties':{'status':{'type':'string','enum':['healthy','recovered','blocked','completed']},
        'summary':{'type':'string'},'repairs':{'type':'array','items':{'type':'string'}}},
        'required':['status','summary','repairs'],'additionalProperties':False}
    atomic_json(folder/'response_schema.json',schema)
    prompt=f"""You are the hourly caretaker for a user-authorized overnight FMM-JEPA research study.
User explicitly authorized the full 12-hour run, hourly checks, operational bug fixes and exact resume while asleep.
Project root {root}. Run ID {run_id}. Event {event}. Snapshot {json.dumps(snap)}.
Read /home/benjamin/workspace/ahr_go/AGENTS.md. All authored executable project code MUST be .ipynb notebooks only under the workspace.
Do not touch the unrelated AHR session or GPU1 jobs. Never kill all GPU/Python processes.
Check study_status, latest checkpoint manifests and the worker log for this exact run. Normal progress requires no changes.
If worker crashed or stalled, diagnose and implement the smallest validated operational recovery then resume from existing valid full-state checkpoints.
Do not alter seeds, model architecture, JEPA objective, optimizer/scheduler/EMA, masks, data membership, quality endpoints or original clock. No skipping batches, replacing missing arms, resetting/retraining completed arms, or silent restarts.
Do not edit any registered export source from study_binding.json or rebind its hash map. Put repairs in new tagged notebook definitions under recoveries/ or validation notebooks, with separate immutable recovery provenance.
A runtime adapter may fix diagnostics, device placement, serialization, resource handling or presentation only if it preserves scientific computation and passes an appropriate smoke/recovery test.
Use same worker tmux session fmm-manifold-long-{run_id} so the independent deadline watcher still owns it. Keep one writer. Stop writer before installing an adapter, and document recovery hashes.
All checkpoints, config, every RNG, consumed sample/mask cursor, optimizer, scheduler, teacher/scaler must remain consistent. Invalid manifests or true numerical trajectory failures must stop as blocked/incomplete, never silently reset.
Main launcher is notebooks/11_Overnight_Manifold_JEPA.ipynb, its ahr-export bootstrap loads WORKFLOW. Unmodified WORKFLOW.launch_study(ROOT, saved_configuration) reconnects/resumes valid states. Make sure any repair is actually active in a detached worker, not only an interactive test.
Absolute deadline {snap['clock'].get('deadline_unix')}; current remaining seconds {remaining}. Do not extend the deadline. Leave the independent hourly supervisor and deadline watcher running.
No services/publication/external messages. Your logs and final JSON are the hourly review. If impossible within the fixed scientific protocol, record blocked with specific cause.
Return the schema with status, concise summary and repair artifact paths. Do not read/print credentials."""
    (folder/'prompt.txt').write_text(prompt)
    command=[str(CLI),'exec','--ignore-user-config','--ephemeral','--skip-git-repo-check',
        '--sandbox','danger-full-access','-c','approval_policy="never"','--json',
        '--output-schema',str(folder/'response_schema.json'),'--output-last-message',str(folder/'answer.json'),'-']
    timeout=min(900,max(1,remaining-10));started=time.time()
    with (folder/'events.jsonl').open('w') as log:
        child=subprocess.Popen(command,stdin=subprocess.PIPE,stdout=log,stderr=subprocess.STDOUT,
            text=True,cwd=root,start_new_session=True)
        try:
            child.communicate(prompt,timeout=timeout);returncode=child.returncode
        except subprocess.TimeoutExpired:
            os.killpg(child.pid,signal.SIGKILL);child.communicate();returncode=-9
    try:answer=_read(folder/'answer.json') if (folder/'answer.json').exists() else {}
    except (json.JSONDecodeError,OSError):answer={}
    result={'status':'PASS' if returncode==0 and answer else 'AGENT_REVIEW_FAILED','event':event,
        'returncode':returncode,'elapsed_seconds':time.time()-started,'answer':answer,
        'answer_path':str(folder/'answer.json'),'events_path':str(folder/'events.jsonl')}
    atomic_json(folder/'review_result.json',result)
    return result
def _restart(root,run_id,number):
    workflow=_workflow(root);artifact=Path(root)/'artifacts/notebook_11'/run_id
    binding=_read(artifact/'study_binding.json')
    workflow._check_binding(root,binding,binding['configuration'])
    # The bound launcher performs exact manifest checks; no new run/config/clock.
    result=workflow.launch_study(root,binding['configuration'])
    atomic_json(artifact/'heartbeat'/f'restart_{number:02d}.json',{'observed_unix':time.time(),'result':result})
    return result
def run_heartbeat(root,run_id):
    root=Path(root);artifact=root/'artifacts/notebook_11'/run_id
    binding=_read(artifact/'study_binding.json');clock=_read(artifact/'budget_clock.json')
    if clock!=binding['budget_clock']:raise ValueError('Heartbeat original clock differs')
    config=binding['configuration']['heartbeat'];folder=artifact/'heartbeat';folder.mkdir(exist_ok=True)
    saved=_read(folder/'supervisor_state.json')
    next_review=float(saved.get('next_review_unix',clock['started_unix']+config['interval_seconds']))
    restarts=int(saved.get('restarts',0));last_repair_unix=float(saved.get('last_repair_unix',0))
    with run_writer(folder/'supervisor_lock'):
        while True:
            snap=snapshot(root,run_id);now=time.time()
            atomic_json(folder/'latest_heartbeat.json',snap)
            if snap['status'].get('status') in TERMINAL or now>=clock['deadline_unix']:
                result={'status':'EXITED_TERMINAL_OR_DEADLINE','study_status':snap['status'],
                    'observed_unix':now,'restarts':restarts,'deadline_unix':clock['deadline_unix']}
                atomic_json(folder/'supervisor_result.json',result);return result
            repair_needed=(not snap['worker_alive'] and snap['status'].get('status')!='STARTING')
            stalled=snap['worker_alive'] and snap['activity_age_seconds']>config['stall_seconds']
            hourly_due=now>=next_review
            if hourly_due or ((repair_needed or stalled) and now-last_repair_unix>300):
                event='worker_failure' if repair_needed else 'worker_stall' if stalled else 'hourly'
                review=agent_review(root,run_id,event)
                atomic_json(folder/f'review_{int(now):010d}.json',review)
                if hourly_due:
                    while next_review<=time.time():next_review+=config['interval_seconds']
                if repair_needed or stalled:last_repair_unix=time.time()
                snap=snapshot(root,run_id)
                # If an agent did not resume an ordinary interrupted worker, use original exact-resume launcher.
                blocked=(review.get('answer') or {}).get('status')=='blocked'
                if blocked:last_repair_unix=time.time()+3300
                if not blocked and not snap['worker_alive'] and snap['status'].get('status') not in TERMINAL:
                    if restarts<config['automatic_resume_attempts'] and time.time()<clock['deadline_unix']-60:
                        restarts+=1
                        try:_restart(root,run_id,restarts)
                        except BaseException as error:
                            atomic_json(folder/f'restart_{restarts:02d}_failure.json',{'error':repr(error),'observed_unix':time.time()})
                atomic_json(folder/'supervisor_state.json',{'next_review_unix':next_review,'restarts':restarts,
                    'last_repair_unix':last_repair_unix,'updated_unix':time.time()})
            time.sleep(min(config['poll_seconds'],max(.1,clock['deadline_unix']-time.time())))
def launch_heartbeat(root,run_id):
    root=Path(root);artifact=root/'artifacts/notebook_11'/run_id;folder=artifact/'heartbeat';folder.mkdir(exist_ok=True)
    session='fmm-manifold-long-heartbeat-'+run_id
    if _exists(session):return {'status':'EXISTING_SESSION','session':session}
    if _read(artifact/'study_status.json').get('status')=='COMPLETED':return {'status':'COMPLETED'}
    module=root/'experiments/11_manifold_long/heartbeat.ipynb'
    code="from pathlib import Path\nimport json\nroot=Path("+repr(str(root))+")\nns={}\nfor c in json.loads((root/'fmm_jepa/bootstrap.ipynb').read_text())['cells']:\n    if c['cell_type']=='code' and 'ahr-export' in c.get('metadata',{}).get('tags',[]):exec(''.join(c['source']),ns)\nns['install_notebook_package'](root)\nfor c in json.loads(Path("+repr(str(module))+").read_text())['cells']:\n    if c['cell_type']=='code' and 'ahr-export' in c.get('metadata',{}).get('tags',[]):exec(compile(''.join(c['source']),'heartbeat#ahr-export','exec'),ns)\nns['run_heartbeat'](root,"+repr(run_id)+")\n"
    worker=folder/'supervisor.ipynb'
    if worker.exists():
        if [c.source for c in nbformat.read(worker,as_version=4).cells if c.cell_type=='code']!=[code]:raise ValueError('Supervisor source changed')
    else:
        nbformat.write(nbformat.v4.new_notebook(cells=[nbformat.v4.new_markdown_cell('# Overnight hourly agent supervisor\nOriginal clock; safe checkpoint resume only; no unrelated sessions.'),nbformat.v4.new_code_cell(code)],
            metadata={'kernelspec':{'display_name':'AHR Go (.venv)','language':'python','name':'ahr-go'}}),worker)
    command=[sys.executable,'-m','jupyter','nbconvert','--execute','--to','notebook','--inplace',
        '--ExecutePreprocessor.timeout=-1','--ExecutePreprocessor.kernel_name=ahr-go',str(worker)]
    shell='exec '+shlex.join(command)+' > '+shlex.quote(str(folder/'supervisor.log'))+' 2>&1'
    subprocess.run(['tmux','new-session','-d','-s',session,'-c',str(root),'bash','-c',shell],check=True)
    record={'status':'LAUNCHED','session':session,'interval_seconds':3600,'poll_seconds':30,
        'deadline_unix':_read(artifact/'budget_clock.json')['deadline_unix'],
        'agent_authenticated_probe':'artifacts/notebook_11_validation/agent_probe/probe_result.json',
        'attach':'tmux attach -t '+session,'detach':'Ctrl-b, then d'}
    atomic_json(folder/'launch.json',record);return record
